# GPU で Qwen3 日本語チャット

Colab の GPU で [Qwen3-1.7B](https://huggingface.co/Qwen/Qwen3-1.7B) を動かします。モデルは Apache-2.0 ライセンスです。初回は数 GB のモデルをダウンロードするため、時間と空き容量が必要です。上から順にセルを実行してください。

まず **ランタイム → ランタイムのタイプを変更 → GPU** を選んでください。このノートブックは外部APIの鍵を使いません。

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU が見つかりません。Colab の「ランタイム → ランタイムのタイプを変更」で GPU を選んでから再実行してください。')
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)
print('空きVRAM: %.1f GiB' % (torch.cuda.mem_get_info()[0] / 1024**3))


## ライブラリを準備

In [ ]:
%pip -q install "transformers>=4.51,<5" accelerate safetensors


## モデルを読み込む

Colab の一般的な GPU を想定し、16-bit で読み込みます。VRAM が足りない場合は他の GPU 作業を終了し、ランタイムを再起動してください。

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'Qwen/Qwen3-1.7B'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to('cuda')
model.eval()
print('読み込み完了:', MODEL_ID)


## 質問する

`QUESTION` を編集してセルを再実行できます。`enable_thinking=False` で短い応答向けにしています。回答は事実確認が必要な場合があります。

In [ ]:
QUESTION = 'Unity の Prefab を初心者向けに説明して'
MAX_NEW_TOKENS = 256  # 長くすると VRAM と時間が増えます

if not QUESTION.strip():
    raise ValueError('QUESTION に質問を入力してください。')
messages = [{'role': 'user', 'content': QUESTION}]
prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
)
inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
with torch.inference_mode():
    outputs = model.generate(
        **inputs, max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True, temperature=0.7, top_p=0.8, top_k=20,
        pad_token_id=tokenizer.eos_token_id,
    )
answer = tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()
print(answer)
